# Functions and Tools

Tools give your agent capabilities to interact with the world. This tutorial covers different ways to add tools to your agents.

## What You'll Learn

1. Built-in tools (CurrentTimeTool, etc.)
2. Function groups (collections of related tools)
3. MCP Client (Model Context Protocol)
4. A2A Client (Agent-to-Agent protocol)
5. Tool configuration options

## Tool Types Overview

| Type | Description | Use Case |
|------|-------------|----------|
| **NatFunction** | Single function/tool | Simple operations |
| **NatFunctionGroup** | Collection of related tools | Related operations |
| **MCPClient** | Connect to MCP servers | External tool servers |
| **A2AClient** | Connect to other agents | Agent orchestration |


In [ ]:
import sys
from pathlib import Path

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


## 1. Built-in Tools

NAT includes several built-in tools you can use immediately:


In [ ]:
from nat.tool.datetime_tools import CurrentTimeTool

# CurrentTimeTool - Returns current date and time
time_tool = CurrentTimeTool(
    name="current_time",
    # Optional: customize timezone
    # timezone="America/New_York"
)

print(f"✅ Created: {time_tool.computed_name}")


## 2. Function Groups

Function groups bundle related tools together. The calculator example demonstrates this:


In [ ]:
# Function groups from installed example packages
try:
    from nat_simple_calculator.register import CalculatorToolGroup

    calculator = CalculatorToolGroup(
        name="calculator",
        # Can specify which functions to include
        include=["add", "subtract", "multiply", "divide"],
    )
    print("✅ Calculator group created with functions: add, subtract, multiply, divide")
except ImportError:
    print("⚠️  Install with: uv pip install -e examples/getting_started/simple_calculator")


## 3. MCP Client (Model Context Protocol)

MCP allows you to connect to external tool servers. This is useful for:
- Using tools from other services
- Connecting to local MCP servers
- Accessing authenticated APIs


In [ ]:
from pydantic import HttpUrl

from nat.plugins.mcp.client_config import MCPClient
from nat.plugins.mcp.client_config import MCPServerConfig
from nat.plugins.mcp.client_config import MCPToolOverrideConfig

# Example 1: Local MCP server (stdio transport)
mcp_time = MCPClient(
    server=MCPServerConfig(
        transport="stdio",
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=America/Los_Angeles"],
    ),
    tool_overrides={
        "get_current_time": MCPToolOverrideConfig(
            alias="get_time_mcp",  # Rename the tool
            description="Get current time from MCP server",
        ),
    },
    name="mcp_time",
)

print("✅ MCP Client (stdio) configured")


In [ ]:
# Example 2: Remote MCP server (HTTP transport)
mcp_remote = MCPClient(
    server=MCPServerConfig(
        transport="streamable-http",
        url=HttpUrl("http://localhost:9901/mcp"),  # NAT MCP server
    ),
    include=["calculator.add", "calculator.multiply"],  # Only include specific tools
    name="mcp_remote",
)

print("✅ MCP Client (HTTP) configured")


## 4. A2A Client (Agent-to-Agent)

A2A allows your agent to use other agents as tools. This enables sophisticated agent orchestration:


In [ ]:
from datetime import timedelta

try:
    from nat.plugins.a2a.client.a2a_client import A2AClient

    # Connect to an A2A agent server
    calculator_agent = A2AClient(
        url=HttpUrl("http://localhost:10000"),
        task_timeout=timedelta(seconds=60),
        include_skills_in_description=True,  # Include agent's capabilities
        name="calculator_agent",
    )
    print("✅ A2A Client configured")
except ImportError:
    print("⚠️  Install with: uv pip install -e packages/nvidia_nat_a2a")


## 5. Combining Tools in an Agent


In [ ]:
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    name="nim_llm",
)

# Combine different tool types
tools = [
    time_tool,      # Built-in function
    mcp_time,       # MCP client
]

agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
)

workflow = NatWorkflow(entrypoint=agent)
print(f"✅ Agent created with {len(tools)} tools")


In [ ]:
# Save configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "tools_example.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Saved to: {config_path}")
print("\n" + "=" * 50 + "\n")
with open(config_path) as f:
    print(f.read())


## CLI Commands

```bash
# Run the workflow
nat run --config_file configs/tools_example.yaml --input "What time is it?"

# Start an MCP server to connect to
nat mcp serve --config_file examples/getting_started/simple_calculator/configs/config.yml

# Start an A2A server to connect to
nat a2a serve --config_file examples/getting_started/simple_calculator/configs/config.yml --port 10000
```

## Summary

✅ **Built-in tools** - Ready-to-use functions like CurrentTimeTool  
✅ **Function groups** - Collections of related tools  
✅ **MCP Client** - Connect to external MCP servers  
✅ **A2A Client** - Use other agents as tools  

## Next Steps

- **[05_llms.ipynb](./05_llms.ipynb)** - Configure different LLM providers
- **[06_memory.ipynb](./06_memory.ipynb)** - Add memory to your agents
